In [1]:
import pandas as pd
from io import StringIO
from sklearn.model_selection import train_test_split

# Load data

csv = """
customer_id,age,income_k,loan_k,years_with_bank,missed_payments,default
C01,23,4.5,1.0,1,0,0
C02,45,12.0,6.0,12,1,0
C03,31,6.0,4.0,5,2,1
C04,26,5.0,2.5,2,0,0
C05,39,9.0,7.5,7,3,1
C06,52,15.0,5.0,18,0,0
C07,29,5.5,3.0,3,1,0
C08,34,7.0,6.5,6,2,1
C09,28,6.5,2.0,4,0,0
C10,41,10.0,8.0,10,4,1
"""

df = pd.read_csv(StringIO(csv))

#
# ============================================================
# EDA CHECKLIST
# ============================================================
#

# 1. Shape and missing values

print("*" * 60)
print("1. DATASET OVERVIEW")
print("*" * 60)

print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

print("Missing values:")
print(df.isna().sum())

print()


# 2. Target distribution

print("*" * 60)
print("2. TARGET DISTRIBUTION")
print("*" * 60)

print(df['default'].value_counts(normalize=True).mul(100).round(2).astype(str) + "%")

print()


# 3. Numeric summary

print("*" * 60)
print("3. NUMERIC SUMMARY STATISTICS")
print("*" * 60)

# For older pandas versions, select numeric columns first

numeric_df = df.select_dtypes(include=['float64', 'int64'])

print(numeric_df.describe().round(2))

print()


# 4. Outlier scan (using IQR method)

print("*" * 60)
print("4. OUTLIER SCAN (IQR Method)")
print("*" * 60)

numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)

    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR

    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    outlier_count = len(outliers)

    outlier_pct = (outlier_count / len(df)) * 100

    print(f"{col:20s}: {outlier_count:2d} outliers ({outlier_pct:.1f}%)")

print()


# 5. Leakage scan

print("*" * 60)

print("5. FEATURE LEAKAGE CHECK")

print("*" * 60)

print(f"Target column: 'default'")

features = [col for col in df.columns if col != 'default' and col != 'customer_id']

print(f"Features: {', '.join(features)}")

print("✓ No target leakage detected (target column excluded from features)")

print()


#
# ============================================================
# TRAIN/TEST SPLIT
# ============================================================
#

# Define features and target (exclude customer_id as it's an identifier)

X = df[['age', 'income_k', 'loan_k', 'years_with_bank', 'missed_payments']]

y = df['default']


# Split with stratification to maintain class distribution

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("*" * 60)

print("TRAIN/TEST SPLIT RESULTS")

print("*" * 60)

print(f"Training set size: {len(X_train)} samples ({len(X_train)/len(df)*100:.0f}%)")

print(f"Test set size: {len(X_test)} samples ({len(X_test)/len(df)*100:.0f}%)\n")


print("Training set class distribution:")

print(y_train.value_counts(normalize=True).mul(100).round(2).astype(str) + "%")

print()


print("Test set class distribution:")

print(y_test.value_counts(normalize=True).mul(100).round(2).astype(str) + "%")

print("\n✓ Stratified split preserved class balance")

************************************************************
1. DATASET OVERVIEW
************************************************************
Shape: 10 rows, 7 columns

Missing values:
customer_id        0
age                0
income_k           0
loan_k             0
years_with_bank    0
missed_payments    0
default            0
dtype: int64

************************************************************
2. TARGET DISTRIBUTION
************************************************************
default
0    60.0%
1    40.0%
Name: proportion, dtype: object

************************************************************
3. NUMERIC SUMMARY STATISTICS
************************************************************
         age  income_k  loan_k  years_with_bank  missed_payments  default
count  10.00     10.00   10.00            10.00            10.00    10.00
mean   34.80      8.05    4.55             6.80             1.30     0.40
std     9.24      3.41    2.42             5.22             1.42     0.5